<div dir="rtl" align="right">
<h2>مقایسهٔ Cross-Entropy معمولی و Cost-Sensitive Loss</h2>
<p>این آزمایش اثر وزن‌دهی در loss را می‌سنجد: آیا تشخیص بدخیمی بهتر می‌شود و در مقابل، خطای مثبت کاذب چه تغییری می‌کند؟ نتیجهٔ مثبت، منفی یا بدون تفاوت، همگی باید گزارش شوند.</p>
<p>GPU را فعال کن، <span dir="ltr">Run all</span> را بزن و Google Drive را وصل کن. تمام کد داخل همین نوت‌بوک است.</p>
<p>دو حالت با همان مدل کامل و یک وزن اولیهٔ مشترک مقایسه می‌شوند: <code dir="ltr">plain_ce</code> با نسبت وزن <code dir="ltr">1:1:1</code> و <code dir="ltr">cost_sensitive_ce</code> با نسبت <code dir="ltr">1:1:2</code> برای کلاس‌های normal، benign و malignant. وزن‌ها با میانگین برچسب‌های train نرمال می‌شوند تا میانگین وزن برابر یک باشد.</p>
<p>برای هر یک از seedهای 42، 43 و 44، هر دو حالت اجرا می‌شوند؛ مجموعاً شش آموزش، با حداکثر ۳۰ epoch و توقف زودهنگام. معماری، learning rate، loss سگمنتیشن، سازوکار وزن‌دهی وظایف، ترتیب داده‌ها، flipها و معیار انتخاب checkpoint یکسان هستند. تنها نسبت وزن کلاس‌ها تغییر می‌کند.</p>
<p>نقطهٔ شروع، همان مدل اولیهٔ قبلاً آموزش‌دیدهٔ پروژه است. پس این مقایسه اثر cost loss را در مرحلهٔ آموزش مدل پیشنهادی بررسی می‌کند. هیچ تصویر test خوانده نمی‌شود.</p>
</div>


In [ ]:
#@title ۱. اتصال به Drive و بررسی GPU
from pathlib import Path
import tensorflow as tf
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


In [ ]:
#@title ۲. تعریف توابع مستقل
from pathlib import Path


import hashlib


import json


import math


import os


import platform


import numpy as np


import pandas as pd


from PIL import Image


import tensorflow as tf


@tf.keras.utils.register_keras_serializable(package="BUSI_MTL")
class SpatialPositionEmbedding(tf.keras.layers.Layer):
    def build(self, input_shape):
        self.positions = self.add_weight(
            name="positions", shape=(1, int(input_shape[1]), int(input_shape[2])),
            initializer=tf.keras.initializers.RandomNormal(stddev=0.02), trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        return inputs + self.positions


@tf.keras.utils.register_keras_serializable(package="BUSI_MTL")
class ProbabilityToLogits(tf.keras.layers.Layer):
    def __init__(self, binary=False, epsilon=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.binary, self.epsilon = binary, epsilon

    def call(self, inputs):
        p = tf.clip_by_value(inputs, self.epsilon, 1.0-self.epsilon)
        return tf.math.log(p)-tf.math.log1p(-p) if self.binary else tf.math.log(p)

    def get_config(self):
        return {**super().get_config(), "binary": self.binary, "epsilon": self.epsilon}


@tf.keras.utils.register_keras_serializable(package="BUSI_Training")
class BoundedLogWeight(tf.keras.constraints.Constraint):
    def __call__(self, value):
        return tf.clip_by_value(value, -3.0, 3.0)

    def get_config(self):
        return {}


def soft_dice_parts(y_true, y_pred):
    y_true, y_pred = tf.cast(y_true, tf.float32), tf.cast(y_pred, tf.float32)
    true_area = tf.reduce_sum(y_true, axis=(1, 2, 3))
    predicted_area = tf.reduce_sum(y_pred, axis=(1, 2, 3))
    intersection = tf.reduce_sum(y_true*y_pred, axis=(1, 2, 3))
    dice = (2*intersection+1e-6)/(true_area+predicted_area+1e-6)
    return dice, tf.cast(true_area > 0, tf.float32)


def segmentation_loss_per_image(y_true, y_pred):
    y_true, y_pred = tf.cast(y_true, tf.float32), tf.cast(y_pred, tf.float32)
    bce = tf.reduce_mean(
        tf.keras.losses.binary_crossentropy(y_true, y_pred), axis=(1, 2))
    dice, positive = soft_dice_parts(y_true, y_pred)
    return bce + positive*(1-dice)


@tf.keras.utils.register_keras_serializable(package="BUSI_Training")
class ProposedTrainer(tf.keras.Model):
    """Explicitly track task weights as trainable MODEL variables, not loss locals."""
    def __init__(self, network, class_weights, adaptive_task_weights=True, **kwargs):
        super().__init__(**kwargs)
        self.network = network
        self.class_weights = [float(x) for x in class_weights]
        self.adaptive_task_weights = bool(adaptive_task_weights)
        self.log_class = self.add_weight(
            name="log_class", shape=(), initializer="zeros", trainable=self.adaptive_task_weights,
            constraint=BoundedLogWeight())
        self.log_seg = self.add_weight(
            name="log_seg", shape=(), initializer="zeros", trainable=self.adaptive_task_weights,
            constraint=BoundedLogWeight())
        self.total_metric = tf.keras.metrics.Mean(name="loss")
        self.class_metric = tf.keras.metrics.Mean(name="classification_loss")
        self.plain_ce_metric = tf.keras.metrics.Mean(name="classification_plain_ce")
        self.seg_metric = tf.keras.metrics.Mean(name="segmentation_loss")
        self.accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy(name="classification_accuracy")
        self.dice_metric = tf.keras.metrics.Mean(name="positive_soft_dice")

    @property
    def metrics(self):
        return [self.total_metric, self.class_metric, self.plain_ce_metric,
                self.seg_metric, self.accuracy_metric, self.dice_metric]

    def build(self, input_shape):
        if not self.network.built:
            self.network.build(input_shape)
        super().build(input_shape)

    def call(self, inputs, training=False):
        return self.network(inputs, training=training)

    def _terms(self, targets, predictions):
        labels = tf.cast(tf.reshape(targets["classification"], [-1]), tf.int32)
        ce = tf.keras.losses.sparse_categorical_crossentropy(labels, predictions["classification"])
        weighted_ce = ce*tf.gather(tf.constant(self.class_weights, tf.float32), labels)
        seg = segmentation_loss_per_image(targets["segmentation"], predictions["segmentation"])
        # Chosen adaptive weighting objective; NOT a clinical uncertainty estimate.
        if self.adaptive_task_weights:
            total = (tf.exp(-self.log_class)*tf.reduce_mean(weighted_ce)+self.log_class
                     + tf.exp(-self.log_seg)*tf.reduce_mean(seg)+self.log_seg)
        else:
            total = tf.reduce_mean(weighted_ce)+tf.reduce_mean(seg)
        if self.losses:
            total += tf.add_n(self.losses)
        tf.debugging.assert_all_finite(total, "Non-finite training objective")
        return total, weighted_ce, ce, seg

    def _update(self, targets, predictions, terms):
        total, weighted_ce, ce, seg = terms
        self.total_metric.update_state(total, sample_weight=tf.cast(tf.shape(ce)[0], tf.float32))
        self.class_metric.update_state(weighted_ce)
        self.plain_ce_metric.update_state(ce)
        self.seg_metric.update_state(seg)
        self.accuracy_metric.update_state(targets["classification"], predictions["classification"])
        dice, positive = soft_dice_parts(targets["segmentation"], predictions["segmentation"])
        self.dice_metric.update_state(dice, sample_weight=positive)
        return {metric.name: metric.result() for metric in self.metrics}

    def train_step(self, data):
        images, targets, sample_weight = tf.keras.utils.unpack_x_y_sample_weight(data)
        if sample_weight is not None:
            raise ValueError("Use the recorded class weights; external sample weights are not supported.")
        with tf.GradientTape() as tape:
            predictions = self(images, training=True)
            terms = self._terms(targets, predictions)
        gradients = tape.gradient(terms[0], self.trainable_variables)
        pairs = [(gradient, variable) for gradient, variable in
                 zip(gradients, self.trainable_variables) if gradient is not None]
        self.optimizer.apply_gradients(pairs)
        return self._update(targets, predictions, terms)

    def test_step(self, data):
        images, targets, _ = tf.keras.utils.unpack_x_y_sample_weight(data)
        predictions = self(images, training=False)
        return self._update(targets, predictions, self._terms(targets, predictions))

    def get_config(self):
        return {**super().get_config(),
                "network": tf.keras.utils.serialize_keras_object(self.network),
                "class_weights": self.class_weights,
                "adaptive_task_weights": self.adaptive_task_weights}

    @classmethod
    def from_config(cls, config):
        config = dict(config)
        network = tf.keras.utils.deserialize_keras_object(config.pop("network"))
        return cls(network=network, **config)


def validation_statistics(labels, masks, probabilities):
    """Image-level validation metrics. All class orders: normal, benign, malignant."""
    labels = np.asarray(labels, dtype=np.int32)
    predicted = np.argmax(probabilities["classification"], axis=1)
    matrix = np.zeros((3, 3), dtype=np.int64)
    np.add.at(matrix, (labels, predicted), 1)
    tp = np.diag(matrix).astype(float)
    precision = np.divide(tp, matrix.sum(0), out=np.zeros(3), where=matrix.sum(0)>0)
    recall = np.divide(tp, matrix.sum(1), out=np.zeros(3), where=matrix.sum(1)>0)
    f1 = np.divide(2*precision*recall, precision+recall,
                   out=np.zeros(3), where=precision+recall>0)
    malignant = labels == 2
    normal = labels == 0
    false_malignant = int(((predicted == 2) & ~malignant).sum())
    specificity = 1-false_malignant/int((~malignant).sum())
    true_masks = np.asarray(masks)[..., 0] > 0
    predicted_masks = np.asarray(probabilities["segmentation"])[..., 0] >= 0.5
    true_area = true_masks.sum((1, 2))
    predicted_area = predicted_masks.sum((1, 2))
    intersection = (true_masks & predicted_masks).sum((1, 2))
    positive = true_area > 0
    dice = np.full(len(labels), np.nan)
    dice[positive] = 2*intersection[positive]/(true_area[positive]+predicted_area[positive])
    stats = {
        "val_accuracy": float(tp.sum()/len(labels)),
        "val_macro_f1": float(f1.mean()),
        "val_malignant_recall": float(recall[2]),
        "val_missed_malignant": int((predicted[malignant] != 2).sum()),
        "val_nonmalignant_predicted_malignant": false_malignant,
        "val_nonmalignant_specificity": float(specificity),
        "val_malignant_balanced_accuracy": float((recall[2]+specificity)/2),
        "val_normal_recall": float(recall[0]),
        "val_positive_dice": float(dice[positive].mean()),
        "val_malignant_dice": float(dice[malignant].mean()),
        "val_very_low_dice": int((dice[positive] < 0.1).sum()),
        "val_empty_positive_predictions": int((predicted_area[positive] == 0).sum()),
        "val_normal_with_any_foreground": int((predicted_area[normal] > 0).sum()),
        "val_normal_mean_foreground_percent": float(
            100*predicted_area[normal].mean()/true_masks.shape[1]/true_masks.shape[2]),
        "val_normal_max_foreground_percent": float(
            100*predicted_area[normal].max()/true_masks.shape[1]/true_masks.shape[2])}
    return stats, matrix, dice, predicted, predicted_area


def _write_json(path, value):
    path = Path(path)
    tmp = path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf8")
    os.replace(tmp, path)


def _save_model(model, path):
    path = Path(path)
    tmp = path.with_name(path.stem+".tmp.keras")
    model.save(str(tmp))
    os.replace(tmp, path)


def _selection_tuple(stats):
    # Predeclared: primary malignant-vs-other balanced accuracy; ties: Dice, macro-F1.
    return (stats["val_malignant_balanced_accuracy"],
            stats["val_positive_dice"], stats["val_macro_f1"])


class ValidationCheckpoints(tf.keras.callbacks.Callback):
    def __init__(self, directory, data, patience, initial_epoch):
        super().__init__()
        self.directory, self.data = Path(directory), data
        self.patience, self.wait = patience, 0
        self.initial_epoch = initial_epoch
        self.best = None
        self.progress = self.directory/"progress.json"
        self.best_metadata = self.directory/"best_validation.json"

    def _predict(self):
        probabilities = self.model.network.predict(self.data["X_val"], batch_size=8, verbose=0)
        for values in probabilities.values():
            if not np.isfinite(values).all():
                raise FloatingPointError("Non-finite validation predictions")
        return validation_statistics(self.data["y_val"], self.data["M_val"], probabilities)[0]

    def _save_best(self, stats, epoch_number):
        _save_model(self.model.network, self.directory/"best.keras")
        _save_model(self.model, self.directory/"best_trainer.keras")
        _write_json(self.best_metadata, {"epoch_number": epoch_number, **stats})
        self.best = _selection_tuple(stats)

    def on_train_begin(self, logs=None):
        if self.best_metadata.exists():
            self.best = _selection_tuple(json.loads(self.best_metadata.read_text()))
        else:
            initial = self._predict()
            self._save_best(initial, self.initial_epoch)
            _write_json(self.directory/"initial_validation.json", initial)
        if self.progress.exists():
            progress = json.loads(self.progress.read_text())
            if progress.get("completed_epochs") == self.initial_epoch:
                self.wait = int(progress.get("wait", 0))

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            raise RuntimeError("Epoch logs missing")
        stats = self._predict()
        logs.update(stats)
        logs["task_weight_classification"] = float(tf.exp(-self.model.log_class).numpy())
        logs["task_weight_segmentation"] = float(tf.exp(-self.model.log_seg).numpy())
        if not np.isfinite(float(logs["loss"])):
            raise FloatingPointError("Non-finite objective")
        score = _selection_tuple(stats)
        if score > self.best:
            self._save_best(stats, epoch+1)
            self.wait = 0
        else:
            self.wait += 1
        record = {"epoch": epoch, **{k: float(v) for k, v in logs.items()}}
        _write_json(self.directory/"epochs"/f"{epoch:04d}.json", record)
        _save_model(self.model, self.directory/"last_trainer.keras")
        _write_json(self.progress, {"completed_epochs": epoch+1, "wait": self.wait})
        print("\nValidation: malignant recall {:.3f}, false malignant {}, Dice {:.3f}, task weights {:.3f}/{:.3f}".format(
            stats["val_malignant_recall"], stats["val_nonmalignant_predicted_malignant"],
            stats["val_positive_dice"], logs["task_weight_classification"], logs["task_weight_segmentation"]))
        if self.wait >= self.patience:
            self.model.stop_training = True
            print("Early stopping: no improvement under the declared selection rule.")


def _read_data(proposed_dir, manifest, root, data):
    frames = {split: manifest[manifest["split"] == split].reset_index(drop=True)
              for split in ["train", "val"]}
    labels = {"normal": 0, "benign": 1, "malignant": 2}
    source = json.loads((proposed_dir/"source_config.json").read_text())
    stats = source["normalization"]
    mean, std = np.array(stats["mean"], np.float32), np.array(stats["std"], np.float32)
    if data is None:
        print("Restoring only train/validation images and masks; no previous training is rerun.")
        data = {}
        for split, frame in frames.items():
            images = np.empty((len(frame), 224, 224, 3), np.float32)
            masks = np.empty((len(frame), 224, 224, 1), np.float32)
            for i, row in frame.iterrows():
                with Image.open(root/row["image"]) as image:
                    images[i] = np.asarray(image.convert("RGB").resize((224,224), Image.Resampling.BILINEAR))
                with Image.open(root/row["mask"]) as image:
                    masks[i,...,0] = np.asarray(image.convert("L").resize((224,224), Image.Resampling.NEAREST)) > 0
                if i % 100 == 0:
                    print(split, i+1, "/", len(frame))
            data["X_"+split] = (images-mean)/std
            data["M_"+split] = masks
            data["y_"+split] = frame["class_name"].map(labels).to_numpy(dtype=np.int32)
    else:
        data = dict(data)
        print("Using the train/validation arrays already in memory.")
    for split, frame in frames.items():
        n = len(frame)
        assert data["X_"+split].shape == (n,224,224,3)
        assert data["M_"+split].shape == (n,224,224,1)
        assert np.array_equal(data["y_"+split], frame["class_name"].map(labels).to_numpy())
        assert np.isfinite(data["X_"+split]).all()
        assert np.isin(data["M_"+split], [0,1]).all()
        assert np.array_equal(data["M_"+split].sum((1,2,3)) > 0, data["y_"+split] != 0)
    assert np.allclose(data["X_train"].mean((0,1,2), dtype=np.float64), 0, atol=0.01), "Expected training Z-score inputs."
    return data, frames


import gc


import shutil


import tempfile


import matplotlib.pyplot as plt


REPAIR_NAMES = ["normal", "benign", "malignant"]


REPAIR_FIELDS = ["val_accuracy", "val_macro_f1", "val_malignant_recall",
    "val_nonmalignant_predicted_malignant", "val_malignant_balanced_accuracy",
    "val_positive_dice", "val_malignant_dice"]


def _repair_sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _repair_guard(path, value):
    if path.is_file():
        if json.loads(path.read_text()) != value:
            raise ValueError("این پوشه مربوط به تنظیمات دیگری است؛ خروجی قبلی را حفظ کن: "+str(path))
    else:
        _write_json(path, value)


def _repair_prediction(network, data):
    values = network.predict(data["X_val"],batch_size=8,verbose=0)
    if not isinstance(values,dict) or set(values) != {"classification","segmentation"}:
        raise ValueError("Expected named classification/segmentation outputs")
    p,mask = [np.asarray(values[key]) for key in ["classification","segmentation"]]
    if p.shape != (115,3) or mask.shape != (115,224,224,1):
        raise ValueError("Unexpected validation output shapes")
    if not np.isfinite(p).all() or not np.isfinite(mask).all() or (p<0).any() or (p>1).any() or (mask<0).any() or (mask>1).any():
        raise FloatingPointError("Invalid validation probabilities")
    np.testing.assert_allclose(p.sum(1),1,atol=1e-5,rtol=0)
    return values


def _repair_per_image(frame, data, values, size_cuts):
    stats,matrix,dice,predicted,predicted_area = validation_statistics(data["y_val"],data["M_val"],values)
    truth = data["y_val"]
    p = np.asarray(values["classification"])
    area = data["M_val"].sum((1,2,3))
    result = frame[["image","mask","class_name","group_id"]].copy()
    result["predicted_class"] = [REPAIR_NAMES[i] for i in predicted]
    result["classification_error"] = predicted != truth
    result["confidence"] = p.max(1)
    result["probability_of_true_class"] = p[np.arange(len(truth)),truth]
    for i,name in enumerate(REPAIR_NAMES):
        result["p_"+name] = p[:,i]
    result["reference_pixels"] = area.astype(int)
    result["reference_area_percent"] = area*100/(224*224)
    result["predicted_pixels"] = predicted_area
    result["dice"] = dice
    result["normal_false_foreground_percent"] = np.where(truth==0,predicted_area*100/(224*224),np.nan)
    result["normal_class_with_mask"] = (predicted==0) & (predicted_area>0)
    result["lesion_class_without_mask"] = (predicted!=0) & (predicted_area==0)
    stats["val_normal_class_with_mask"] = int(result["normal_class_with_mask"].sum())
    stats["val_lesion_class_without_mask"] = int(result["lesion_class_without_mask"].sum())
    stats["val_inconsistent_pairs"] = stats["val_normal_class_with_mask"]+stats["val_lesion_class_without_mask"]
    bins = np.searchsorted(size_cuts,area/(224*224),side="right")
    result["lesion_size_bin"] = ["no_lesion" if a==0 else "Q"+str(int(b)+1) for a,b in zip(area,bins)]
    return stats,matrix,result


def _repair_save_predictions(folder, frame, data, values, size_cuts, condition, seed, epoch):
    folder.mkdir(parents=True,exist_ok=True)
    stats,matrix,per_image = _repair_per_image(frame,data,values,size_cuts)
    per_image.to_csv(folder/"validation_per_image.csv",index=False)
    pd.DataFrame(matrix,index=REPAIR_NAMES,columns=REPAIR_NAMES).to_csv(folder/"validation_confusion.csv")
    binary = np.asarray(values["segmentation"])[...,0] >= 0.5
    temporary = folder/"validation_predictions.tmp.npz"
    np.savez_compressed(temporary,probabilities=np.asarray(values["classification"],np.float32),
        mask_bits=np.packbits(binary.reshape(len(binary),-1),axis=1),
        images=frame["image"].to_numpy(dtype=str),mask_shape=np.array([224,224],np.int32))
    os.replace(temporary,folder/"validation_predictions.npz")
    row = {"condition":condition,"seed":seed,"selected_epoch":int(epoch),**stats}
    _write_json(folder/"summary.json",row)
    return row,per_image


def _repair_check_baseline_config(config, initial_sha, weights):
    expected = {"source_sha256":initial_sha,"class_order":REPAIR_NAMES,
        "class_cost_ratio":[1,1,2],"cnn_encoder_trainable":False,
        "optimizer":"Adam","learning_rate":1e-4,"clipnorm":1.0,
        "batch_size":8,"max_epochs":30,"patience":8,
        "augmentation":"paired_horizontal_flip_probability_0.5",
        "selection_priority":["malignant_vs_others_balanced_accuracy","positive_hard_dice","macro_f1"],
        "adaptive_loss":"exp(-s_cls)*weighted_CE+s_cls+exp(-s_seg)*seg_loss+s_seg",
        "adaptive_log_weight_bounds":[-3,3],"initial_task_weights":[1,1],
        "segmentation_loss":"BCE_all_plus_Dice_on_positive_images"}
    different = {k:(config.get(k),v) for k,v in expected.items() if config.get(k) != v}
    if different:
        raise ValueError("تنظیمات کنترل با این آزمایش همسان نیست: "+str(different))
    np.testing.assert_allclose(config["normalized_class_weights"],weights,atol=1e-6,rtol=0)


def _repair_choose_protocol(experiments_root, evaluation_dir):
    if evaluation_dir is not None:
        return Path(evaluation_dir)
    found = sorted(Path(experiments_root).glob("proposed_model_*/ablations/component_study_v1/final_evaluation_v1/frozen_protocol.json"))
    if not found:
        raise FileNotFoundError("پروژهٔ ارزیابی نهایی قبلی پیدا نشد: "+str(experiments_root))
    for i,path in enumerate(found):
        print(f"[{i}] {path.parent}")
    answer = "0" if len(found)==1 else input("شمارهٔ پروژهٔ قبلی را وارد کن: ").strip()
    if not answer.isdigit() or not 0<=int(answer)<len(found):
        raise ValueError("شمارهٔ پروژه معتبر نیست")
    return found[int(answer)].parent


def _phase_leaves(model,prefix=""):
    found=[]
    for layer in model.layers:
        path=prefix+layer.name
        if isinstance(layer,tf.keras.Model):
            found.extend(_phase_leaves(layer,path+"/"))
        else:
            found.append((path,layer))
    return found


def _phase_frozen_digest(network,allowed):
    digest=hashlib.sha256()
    for path,layer in _phase_leaves(network.get_layer("shared_encoder")):
        if path in allowed:
            continue
        for weight in layer.weights:
            array=np.ascontiguousarray(weight.numpy())
            digest.update(str((path,weight.name,array.shape,array.dtype.str)).encode())
            digest.update(memoryview(array).cast("B"))
    return digest.hexdigest()


def _phase_epoch_arrays(data,seed,epoch):
    """Identical shuffling and flips for both conditions in the same seed/epoch."""
    n=len(data["y_train"])
    order=np.random.default_rng(np.random.SeedSequence([seed,epoch,4801])).permutation(n)
    flags=np.random.default_rng(np.random.SeedSequence([seed,epoch,4802])).random(n)<.5
    x,m=data["X_train"][order].copy(),data["M_train"][order].copy()
    x[flags]=x[flags,:,::-1,:]
    m[flags]=m[flags,:,::-1,:]
    return x,m,data["y_train"][order]


from sklearn.metrics import roc_auc_score
FINAL_NAMES=REPAIR_NAMES.copy()

def _final_metrics(labels, masks, probabilities, prefix="val", extended=False):
    labels = np.asarray(labels, np.int32)
    p = np.asarray(probabilities["classification"], np.float64)
    soft_masks = np.asarray(probabilities["segmentation"])
    if (p.shape != (len(labels),3) or soft_masks.shape != np.shape(masks) or
            not np.isfinite(p).all() or not np.isfinite(soft_masks).all() or
            ((p < 0) | (p > 1)).any() or ((soft_masks < 0) | (soft_masks > 1)).any()):
        raise ValueError("Invalid class or mask probabilities")
    np.testing.assert_allclose(p.sum(1), 1, atol=1e-5, rtol=0)
    p = p/p.sum(1, keepdims=True)
    predicted = p.argmax(1)
    cm = np.zeros((3,3), np.int64)
    np.add.at(cm, (labels,predicted), 1)
    tp = np.diag(cm).astype(float)
    precision = np.divide(tp, cm.sum(0), out=np.zeros(3), where=cm.sum(0)>0)
    recall = np.divide(tp, cm.sum(1), out=np.zeros(3), where=cm.sum(1)>0)
    f1 = np.divide(2*precision*recall, precision+recall,
                   out=np.zeros(3), where=precision+recall>0)
    malignant, normal = labels == 2, labels == 0
    false_malignant = int(((predicted == 2) & ~malignant).sum())
    specificity = 1-false_malignant/int((~malignant).sum())
    truth = np.asarray(masks)[...,0] > 0
    binary = soft_masks[...,0] >= 0.5
    true_area, predicted_area = truth.sum((1,2)), binary.sum((1,2))
    intersection = (truth & binary).sum((1,2))
    positive = true_area > 0
    dice, iou = np.full(len(labels),np.nan), np.full(len(labels),np.nan)
    dice[positive] = 2*intersection[positive]/(true_area[positive]+predicted_area[positive])
    iou[positive] = intersection[positive]/(true_area[positive]+predicted_area[positive]-intersection[positive])
    normal_with_mask = int(((predicted == 0) & (predicted_area > 0)).sum())
    lesion_without_mask = int(((predicted != 0) & (predicted_area == 0)).sum())
    pixels = truth.shape[1]*truth.shape[2]
    values = {
        "accuracy":float(tp.sum()/len(labels)), "macro_f1":float(f1.mean()),
        "malignant_recall":float(recall[2]), "missed_malignant":int((predicted[malignant] != 2).sum()),
        "nonmalignant_predicted_malignant":false_malignant,
        "nonmalignant_specificity":float(specificity),
        "malignant_balanced_accuracy":float((recall[2]+specificity)/2),
        "normal_recall":float(recall[0]), "positive_dice":float(dice[positive].mean()),
        "malignant_dice":float(dice[malignant].mean()), "positive_iou":float(iou[positive].mean()),
        "very_low_dice":int((dice[positive] < 0.1).sum()),
        "empty_positive_predictions":int((predicted_area[positive] == 0).sum()),
        "normal_with_any_foreground":int((predicted_area[normal] > 0).sum()),
        "normal_mean_foreground_percent":float(100*predicted_area[normal].mean()/pixels),
        "normal_max_foreground_percent":float(100*predicted_area[normal].max()/pixels),
        "inconsistent_pairs":normal_with_mask+lesion_without_mask,
    }
    if extended:
        confidence, correct = p.max(1), predicted == labels
        bins = np.minimum(np.searchsorted(np.linspace(0,1,16),confidence,side="right")-1,14)
        ece = sum(float((bins == index).mean())*
                  abs(float(correct[bins == index].mean())-float(confidence[bins == index].mean()))
                  for index in range(15) if (bins == index).any())
        values.update({
            "malignant_auc":float(roc_auc_score(malignant.astype(int),p[:,2])),
            "macro_auc_ovr":float(roc_auc_score(labels,p,multi_class="ovr",average="macro",labels=[0,1,2])),
            "ece_15bins":float(ece),
            "brier_multiclass_sum":float(((p-np.eye(3)[labels])**2).sum(1).mean()),
            "negative_log_likelihood":float(-np.log(np.clip(p[np.arange(len(labels)),labels],1e-7,1)).mean()),
        })
    stats = {prefix+"_"+key:value for key,value in values.items()}
    per_class = pd.DataFrame({"class_name":FINAL_NAMES,"precision":precision,
                              "recall":recall,"f1":f1,"support":cm.sum(1)})
    return stats, cm, dice, iou, predicted, predicted_area, binary, p, per_class


"""Prospectively declared loss ablation for the proposed multi-task model."""
COST_VERSION="1.0.0"
COST_SEEDS=[42,43,44]
COST_PROFILES={"plain_ce":[1.,1.,1.],"cost_sensitive_ce":[1.,1.,2.]}


def _cost_weights(profile,labels):
    raw=np.asarray(COST_PROFILES[profile],np.float32)
    labels=np.asarray(labels,np.int32)
    if not np.isin(labels,[0,1,2]).all() or len(labels)==0:
        raise ValueError("Invalid training class labels")
    return raw/raw[labels].mean()


def _cost_new_trainer(initial_path,profile,data):
    network=tf.keras.models.load_model(str(initial_path),compile=False)
    network.get_layer("shared_encoder").trainable=False
    weights=_cost_weights(profile,data["y_train"])
    trainer=ProposedTrainer(network,weights.tolist(),name="cost_loss_trainer")
    trainer(data["X_train"][:1],training=False)
    trainer.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4,clipnorm=1.0))
    if not network.trainable_weights:
        raise ValueError("The network has no trainable head/shared-transformer weights")
    np.testing.assert_allclose([float(trainer.log_class.numpy()),float(trainer.log_seg.numpy())],[0,0],atol=0,rtol=0)
    return trainer


def _cost_smoke(initial_path,data,report):
    records=[];previous=None
    indices=[int(np.flatnonzero(data["y_train"]==i)[0]) for i in range(3)]
    x=data["X_train"][indices]
    target={"classification":data["y_train"][indices],"segmentation":data["M_train"][indices]}
    for profile in COST_PROFILES:
        tf.keras.utils.set_random_seed(42)
        trainer=_cost_new_trainer(initial_path,profile,data)
        frozen=_phase_frozen_digest(trainer.network,[])
        prediction=trainer(x,training=False)
        if previous is not None:
            for key in prediction:
                np.testing.assert_allclose(prediction[key].numpy(),previous[key],atol=1e-7,rtol=0)
        previous={key:value.numpy().copy() for key,value in prediction.items()}
        terms=trainer._terms(target,prediction)
        ce=tf.keras.losses.sparse_categorical_crossentropy(target["classification"],prediction["classification"]).numpy()
        weights=_cost_weights(profile,data["y_train"])
        np.testing.assert_allclose(terms[1].numpy(),ce*weights[target["classification"]],atol=1e-6,rtol=1e-6)
        np.testing.assert_allclose(weights[data["y_train"]].mean(),1.,atol=1e-6,rtol=0)
        before=[w.numpy().copy() for w in trainer.network.trainable_weights]
        logs=trainer.train_on_batch(x,target,return_dict=True)
        if not all(np.isfinite(float(v)) for v in logs.values()):
            raise FloatingPointError("Non-finite loss-ablation smoke-test metric")
        if not any(not np.array_equal(a,w.numpy()) for a,w in zip(before,trainer.network.trainable_weights)):
            raise RuntimeError("The smoke test did not update network weights")
        if _phase_frozen_digest(trainer.network,[])!=frozen:
            raise RuntimeError("Frozen CNN weights or BatchNorm changed")
        expected=trainer(x,training=False)
        with tempfile.TemporaryDirectory(prefix="busi_cost_smoke_") as directory:
            path=Path(directory)/"trainer.keras"
            _save_model(trainer,path)
            loaded=tf.keras.models.load_model(str(path))
            observed=loaded(x,training=False)
            for key in expected:
                np.testing.assert_allclose(expected[key].numpy(),observed[key].numpy(),atol=1e-6,rtol=1e-6)
            np.testing.assert_allclose(loaded.class_weights,weights,atol=1e-7,rtol=0)
            if int(loaded.optimizer.iterations.numpy())!=1:
                raise RuntimeError("Optimizer checkpoint restoration failed")
            del loaded,observed
        records.append({"profile":profile,"class_cost_ratio":COST_PROFILES[profile],
            "normalized_class_weights":weights.tolist(),"initial_outputs_match":True,
            "weighted_ce_checked":True,"network_updated":True,"frozen_encoder_unchanged":True,
            "optimizer_save_reload":"passed","logs":{k:float(v) for k,v in logs.items()}})
        del trainer,before,prediction,expected,terms
        gc.collect()
    _write_json(report/"colab_smoke_test.json",records)
    print("Both loss smoke tests passed; original network weights were not modified.",flush=True)


def _cost_record(folder,frame,data,values,cuts,profile,seed,epoch):
    row,table=_repair_save_predictions(folder,frame,data,values,cuts,profile,seed,epoch)
    extended=_final_metrics(data["y_val"],data["M_val"],values,extended=True)[0]
    row.update(extended)
    row["initial_model_retained"]=int(epoch)==0
    _write_json(folder/"summary.json",row)
    return row,table


def _cost_run_one(out,report,initial_path,profile,seed,data,frame,cuts,reference_values):
    directory=out/profile/f"seed{seed}"
    directory.mkdir(parents=True,exist_ok=True);(directory/"epochs").mkdir(exist_ok=True)
    destination=report/f"{profile}_seed{seed}"
    weights=_cost_weights(profile,data["y_train"])
    _repair_guard(directory/"settings.json",{"experiment_settings_sha256":_repair_sha(out/"settings.json"),
        "profile":profile,"seed":seed,"class_cost_ratio":COST_PROFILES[profile],
        "normalized_class_weights":weights.tolist()})
    if (directory/"completed.json").is_file():
        record=json.loads((directory/"completed.json").read_text())
        if _repair_sha(directory/"best.keras")!=record["best_sha256"]:
            raise ValueError("A completed ablation checkpoint changed")
        return json.loads((destination/"summary.json").read_text()),pd.read_csv(destination/"validation_per_image.csv")
    tf.keras.utils.set_random_seed(seed)
    fresh=_cost_new_trainer(initial_path,profile,data)
    initial=_repair_prediction(fresh.network,data)
    for key in initial:
        np.testing.assert_allclose(initial[key],reference_values[key],atol=1e-5,rtol=0)
    frozen=_phase_frozen_digest(fresh.network,[])
    steps=math.ceil(len(data["y_train"])/8)
    last=directory/"last_trainer.keras"
    if last.is_file():
        del fresh;gc.collect()
        trainer=tf.keras.models.load_model(str(last))
        np.testing.assert_allclose(trainer.class_weights,weights,atol=1e-7,rtol=0)
        if trainer.network.get_layer("shared_encoder").trainable or _phase_frozen_digest(trainer.network,[])!=frozen:
            raise ValueError("Resumed encoder state differs")
        iterations=int(trainer.optimizer.iterations.numpy())
        if iterations%steps:
            raise ValueError("Checkpoint is not at an epoch boundary")
        start=iterations//steps
        for epoch in range(start):
            if not (directory/"epochs"/f"{epoch:04d}.json").is_file():
                raise FileNotFoundError("Missing completed epoch record")
        print("Resuming",profile,"seed",seed,"after epoch",start,flush=True)
    else:
        trainer=fresh;start=0
    val=tf.data.Dataset.from_tensor_slices((data["X_val"],{
        "classification":data["y_val"],"segmentation":data["M_val"]})).batch(8)
    monitor=ValidationCheckpoints(directory,data,patience=8,initial_epoch=start)
    for epoch in range(start,30):
        progress=json.loads((directory/"progress.json").read_text()) if (directory/"progress.json").is_file() else {}
        if progress.get("completed_epochs")==epoch and progress.get("wait",0)>=8:
            break
        x,m,y=_phase_epoch_arrays(data,seed,epoch)
        digest=hashlib.sha256()
        for array in [x,m,y]:digest.update(memoryview(np.ascontiguousarray(array)).cast("B"))
        fingerprint=out/"paired_epoch_inputs"/f"seed{seed}_epoch{epoch:02d}.json"
        fingerprint.parent.mkdir(exist_ok=True)
        _repair_guard(fingerprint,{"sha256":digest.hexdigest(),"seed":seed,"epoch_number":epoch+1})
        train=tf.data.Dataset.from_tensor_slices((x,{"classification":y,"segmentation":m})).batch(8).prefetch(1)
        monitor.initial_epoch=epoch
        trainer.fit(train,validation_data=val,initial_epoch=epoch,epochs=epoch+1,callbacks=[monitor],verbose=2)
        if int(trainer.optimizer.iterations.numpy())!=(epoch+1)*steps:
            raise RuntimeError("Unexpected training step count")
        if _phase_frozen_digest(trainer.network,[])!=frozen:
            raise RuntimeError("Frozen encoder state changed during training")
        records=[json.loads((directory/"epochs"/f"{e:04d}.json").read_text()) for e in range(epoch+1)]
        pd.DataFrame(records).to_csv(directory/"training_log.csv",index=False)
        del train,x,m,y;gc.collect()
        if trainer.stop_training:break
    del trainer,val,monitor,initial;gc.collect()
    best=tf.keras.models.load_model(str(directory/"best.keras"),compile=False)
    if _phase_frozen_digest(best,[])!=frozen:
        raise RuntimeError("Selected model changed the frozen encoder")
    values=_repair_prediction(best,data)
    metadata=json.loads((directory/"best_validation.json").read_text())
    metrics=validation_statistics(data["y_val"],data["M_val"],values)[0]
    for field in REPAIR_FIELDS:
        np.testing.assert_allclose(metrics[field],metadata[field],atol=1e-5,rtol=0)
    row,table=_cost_record(destination,frame,data,values,cuts,profile,seed,metadata["epoch_number"])
    for filename in ["settings.json","training_log.csv","best_validation.json","initial_validation.json"]:
        if (directory/filename).is_file():shutil.copyfile(directory/filename,destination/filename)
    _write_json(directory/"completed.json",{"best_sha256":_repair_sha(directory/"best.keras"),"test_evaluated":False})
    del best,values;gc.collect()
    return row,table


def _cost_tables(report,rows,per_images):
    summary=pd.DataFrame(rows)
    summary.to_csv(report/"validation_comparison.csv",index=False)
    fields=["val_accuracy","val_macro_f1","val_malignant_recall","val_missed_malignant",
        "val_nonmalignant_predicted_malignant","val_nonmalignant_specificity","val_malignant_balanced_accuracy",
        "val_positive_dice","val_malignant_dice","val_normal_mean_foreground_percent","val_inconsistent_pairs",
        "val_malignant_auc","val_ece_15bins","val_brier_multiclass_sum","val_negative_log_likelihood"]
    means=summary.groupby("condition",sort=False)[fields].mean()
    means.to_csv(report/"validation_means.csv")
    summary.groupby("condition",sort=False)[fields].agg(["mean","std"]).to_csv(report/"validation_mean_std.csv")
    plain=summary[summary["condition"]=="plain_ce"].set_index("seed")
    weighted=summary[summary["condition"]=="cost_sensitive_ce"].set_index("seed")
    difference=weighted[fields]-plain[fields]
    difference.to_csv(report/"paired_weighted_minus_plain.csv")
    all_images=pd.concat(per_images,ignore_index=True)
    all_images.to_csv(report/"validation_all_images.csv",index=False)
    all_images[all_images["classification_error"]].sort_values("confidence",ascending=False).to_csv(report/"classification_errors.csv",index=False)
    effect={"comparison":"weighted CE minus plain CE; descriptive validation results",
        "mean_differences":{field:float(difference[field].mean()) for field in fields},
        "seeds_with_fewer_missed_malignant":int((difference["val_missed_malignant"]<0).sum()),
        "seeds_with_more_false_malignant":int((difference["val_nonmalignant_predicted_malignant"]>0).sum()),
        "initial_retained_runs":int(summary["initial_model_retained"].sum()),"test_evaluated":False}
    _write_json(report/"cost_effect_summary.json",effect)
    return summary,means


def run_busi_cost_loss_ablation(evaluation_dir=None,
        dataset_root="/content/drive/MyDrive/BUSI_MERGED",experiments_root="/content/drive/MyDrive/BUSI_EXPERIMENTS"):
    if not tf.config.list_physical_devices("GPU"):raise RuntimeError("GPU را در Colab فعال کن.")
    evaluation_dir=_repair_choose_protocol(experiments_root,evaluation_dir)
    protocol_file=evaluation_dir/"frozen_protocol.json"
    protocol=json.loads(protocol_file.read_text())
    if _repair_sha(protocol_file)!=json.loads((evaluation_dir/"completed.json").read_text())["protocol_sha256"]:
        raise ValueError("Frozen evaluation protocol changed")
    if protocol["class_order"]!=REPAIR_NAMES:raise ValueError("Unexpected class order")
    originals=[]
    for seed in COST_SEEDS:
        items=[i for i in protocol["checkpoints"] if i["variant"]=="full_lprm_v1" and i["seed"]==seed]
        if len(items)!=1:raise ValueError("Missing unique original run")
        originals.append(items[0])
    sources={str(Path(i["model"]).parent.parent) for i in originals}
    if len(sources)!=1:raise ValueError("Expected one original project")
    source=Path(sources.pop());initial_path=source/"initial.keras"
    initial_sha=_repair_sha(initial_path)
    manifest_file=source/"split_manifest.csv"
    if _repair_sha(manifest_file)!=protocol["manifest_sha256"]:raise ValueError("Fixed split changed")
    manifest=pd.read_csv(manifest_file)
    if len(manifest)!=761 or not manifest["image"].is_unique or manifest["split"].value_counts().to_dict()!={"train":499,"val":115,"test":147}:
        raise ValueError("Unexpected fixed split")
    if manifest["group_id"].isna().any() or manifest.groupby("group_id")["split"].nunique().max()!=1:
        raise ValueError("Similarity groups cross split boundaries")
    for column in ["image","mask"]:
        if any(Path(p).is_absolute() or ".." in Path(p).parts for p in manifest[column]):raise ValueError("Expected relative paths")
    normalization=json.loads((source/"source_config.json").read_text())["normalization"]
    for key in ["mean","std"]:np.testing.assert_allclose(normalization[key],protocol["normalization"][key],atol=1e-7,rtol=0)
    labels=manifest.loc[manifest["split"]=="train","class_name"].map(dict(zip(REPAIR_NAMES,range(3)))).to_numpy(np.int32)
    anchors=[]
    for item in originals:
        folder=Path(item["model"]).parent
        config_file=folder/"config.json"
        _repair_check_baseline_config(json.loads(config_file.read_text()),initial_sha,_cost_weights("cost_sensitive_ce",labels))
        anchors.append({"seed":item["seed"],"config_sha256":_repair_sha(config_file),
            "initial_validation_sha256":_repair_sha(folder/"initial_validation.json")})
    out=evaluation_dir.parent/"cost_loss_ablation_v1";out.mkdir(exist_ok=True)
    report=out/"reports";report.mkdir(exist_ok=True)
    settings={"version":COST_VERSION,"profiles":COST_PROFILES,"seeds":COST_SEEDS,
        "initial_model":str(initial_path),"initial_model_sha256":initial_sha,"anchors":anchors,
        "frozen_protocol_sha256":_repair_sha(protocol_file),"manifest_sha256":_repair_sha(manifest_file),
        "source_config_sha256":_repair_sha(source/"source_config.json"),"normalization":normalization,
        "class_order":REPAIR_NAMES,"normalized_weights":{p:_cost_weights(p,labels).tolist() for p in COST_PROFILES},
        "loss_difference_only":"training-mean-normalized class cost ratio 1:1:1 versus 1:1:2",
        "segmentation_loss":"unchanged BCE_all + Dice_on_positive_images","adaptive_task_weights":True,
        "initial_task_coefficients":[1,1],"cnn_encoder_trainable":False,"optimizer":"fresh Adam",
        "learning_rate":1e-4,"clipnorm":1.,"batch_size":8,"max_epochs":30,"patience":8,
        "augmentation":"same paired horizontal flips (p=.5) and stateless epoch shuffling in both profiles",
        "selection":["malignant_vs_other_balanced_accuracy","positive_Dice","macro_F1"],
        "initial_checkpoint_included":True,"test_evaluated":False,
        "interpretation":"cost-loss effect during the proposed training stage from a common previously trained warm start; not training from scratch",
        "test_history":"earlier test results already seen; this is an exploratory validation-only ablation, not an independent test",
        "resume":"epoch/optimizer state resumes; dropout replay after interruption is not bit exact",
        "versions":{"tensorflow":tf.__version__,"keras":tf.keras.__version__,"numpy":np.__version__,"python":platform.python_version()}}
    _repair_guard(out/"settings.json",settings);shutil.copyfile(out/"settings.json",report/"settings.json")
    if (out/"completed.json").is_file():
        saved=json.loads((out/"completed.json").read_text())
        for name in saved["report_files"]:
            if not (report/name).is_file():raise FileNotFoundError("Missing report: "+name)
        archive=shutil.make_archive("/content/BUSI_Cost_Loss_RESULTS","zip",root_dir=report)
        print("Completed loss ablation reused; no training rerun.")
        return {"directory":out,"comparison":pd.read_csv(report/"validation_comparison.csv"),"zip_path":archive}
    data,frames=_read_data(source,manifest,Path(dataset_root),None)
    digest=hashlib.sha256()
    for key in ["X_train","M_train","y_train","X_val","M_val","y_val"]:
        a=np.ascontiguousarray(data[key]);digest.update(str((key,a.shape,a.dtype.str)).encode());digest.update(memoryview(a).cast("B"))
    _repair_guard(out/"input_arrays.json",{"sha256":digest.hexdigest()});shutil.copyfile(out/"input_arrays.json",report/"input_arrays.json")
    cuts=np.quantile(data["M_train"].sum((1,2,3))[data["y_train"]!=0]/(224*224),[.25,.5,.75])
    np.savez_compressed(report/"validation_reference.npz",images=frames["val"]["image"].to_numpy(dtype=str),labels=data["y_val"],
        mask_bits=np.packbits((data["M_val"][...,0]>0).reshape(115,-1),axis=1),mask_shape=np.array([224,224]))
    network=tf.keras.models.load_model(str(initial_path),compile=False);network.get_layer("shared_encoder").trainable=False
    reference=_repair_prediction(network,data)
    initial_stats=validation_statistics(data["y_val"],data["M_val"],reference)[0]
    checks=[]
    for item in originals:
        expected=json.loads((Path(item["model"]).parent/"initial_validation.json").read_text())
        for field in REPAIR_FIELDS:
            checks.append({"seed":item["seed"],"metric":field,"expected":expected[field],"observed":initial_stats[field],
                "matches":bool(np.isclose(expected[field],initial_stats[field],atol=1e-5,rtol=0))})
    pd.DataFrame(checks).to_csv(report/"initial_validation_recheck.csv",index=False)
    if not all(i["matches"] for i in checks):raise ValueError("Original initial validation differs; training has not started")
    _write_json(report/"common_initial_validation.json",initial_stats)
    del network;gc.collect()
    if not (report/"colab_smoke_test.json").is_file():_cost_smoke(initial_path,data,report)
    rows,per_images=[],[]
    for seed in COST_SEEDS:
        for profile in COST_PROFILES:
            print("Training",profile,"seed",seed,flush=True)
            row,table=_cost_run_one(out,report,initial_path,profile,seed,data,frames["val"],cuts,reference)
            rows.append(row);table.insert(0,"condition",profile);table.insert(1,"seed",seed);per_images.append(table)
            pd.DataFrame(rows).to_csv(report/"validation_comparison_progress.csv",index=False)
    summary,means=_cost_tables(report,rows,per_images)
    shutil.copytree(out/"paired_epoch_inputs",report/"paired_epoch_inputs",dirs_exist_ok=True)
    notes=("Loss ablation: plain CE (1:1:1) versus cost-sensitive weighted CE (1:1:2), both retrained under one paired protocol.\n"
        "Class costs are normalized to mean 1 over TRAIN labels. They are cost ratios, not inverse-frequency class balancing.\n"
        "Architecture, common warm-start network, task-loss mechanism, optimizer, data order, flips and checkpoint rule are held the same.\n"
        "This measures the loss choice during the proposed training stage; the shared warm start already contains prior task learning.\n"
        "All checkpoints include the initial network; selected epoch 0 means no checkpoint beat that initial state under the selection rule.\n"
        "Both the missed-malignant and false-malignant counts must accompany Recall. Report null or negative results honestly.\n"
        "All evaluation uses the existing 115 validation images. Prior test results have been inspected; this is not an independent test.\n"
        "Weights and optimizer checkpoints stay in Drive; this ZIP includes only the result files.\n"
        "Reference: https://keras.io/api/losses/probabilistic_losses/\n")
    (report/"README.txt").write_text(notes,encoding="utf-8")
    if _repair_sha(initial_path)!=initial_sha:raise RuntimeError("The original initial model changed")
    _write_json(out/"completed.json",{"version":COST_VERSION,"test_evaluated":False,
        "report_files":[str(p.relative_to(report)) for p in sorted(report.rglob("*")) if p.is_file()]})
    archive=shutil.make_archive("/content/BUSI_Cost_Loss_RESULTS","zip",root_dir=report)
    print(means.to_string());print("Saved:",out)
    return {"directory":out,"comparison":summary,"means":means,"zip_path":archive}


In [ ]:
#@title ۳. کنترل، مقایسهٔ loss و دانلود گزارش
from IPython.display import display
from google.colab import files

cost_result = run_busi_cost_loss_ablation()

columns = ["condition", "seed", "selected_epoch", "val_malignant_recall",
           "val_missed_malignant", "val_nonmalignant_predicted_malignant",
           "val_malignant_balanced_accuracy", "val_positive_dice",
           "val_malignant_auc", "val_ece_15bins"]
display(cost_result["comparison"][columns])
print("Saved:", cost_result["directory"])
files.download(cost_result["zip_path"])


<div dir="rtl" align="right">
<p>پیش از آموزش طولانی، تطابق خروجی مدل اولیه با گزارش قبلی، اعمال درست وزن‌های loss، آپدیت شبکه، ثابت ماندن encoder و ذخیره/بارگذاری optimizer در خود Colab بررسی می‌شوند.</p>
<p>در پایان، فایل <code dir="ltr">BUSI_Cost_Loss_RESULTS.zip</code> دانلود می‌شود. این فایل شامل جدول مقایسه، اختلاف دو حالت برای هر seed، خطاهای تک‌تک تصاویر، AUC، ECE و نتایج سگمنتیشن است؛ آن را برای بررسی بفرست.</p>
<p>مدل‌ها در پوشهٔ <code dir="ltr">cost_loss_ablation_v1</code> داخل پروژهٔ قبلی ذخیره می‌شوند. اگر اجرا قطع شد، همین نوت‌بوک را دوباره اجرا کن تا از آخرین checkpoint ادامه دهد. مقدار <code dir="ltr">selected_epoch=0</code> یعنی مدل اولیه به‌عنوان خروجی منتخب حفظ شده است.</p>
<p>این نتایج از همان validation و پس از مشاهدهٔ test قبلی به دست می‌آیند؛ مقایسهٔ اکتشافی هستند و آزمون مستقل جدید محسوب نمی‌شوند. این نوت‌بوک هنوز در محیط محلی روی TensorFlow اجرا نشده؛ آزمون واقعی آن در ابتدای اجرای Colab انجام می‌شود.</p>
</div>
